# EmberWatch — M2 smoke detector training (Kaggle GPU)

Owner: Mouhib · Sprint 1 · CDC target: **smoke mAP@0.5 ≥ 0.55** (stretch ≥ 0.65) on the held-out test set.

**Before running:** Settings → Accelerator → *GPU T4 x2* or *P100*; Internet → *On*. Add D-Fire (and optionally the HPWREN / AI For Mankind smoke boxes) with **Add data**, then set the paths below.

In [ ]:
# 1) Check the GPU and get the code (branch with the M2 module)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -b m2-detection https://github.com/achrefchaabene/EmberWatch.git /kaggle/working/EmberWatch
%cd /kaggle/working/EmberWatch
!pip install -q ultralytics sahi

In [ ]:
# 2) Dataset paths — edit to match the datasets you attached
from pathlib import Path
DFIRE_DIR = Path('/kaggle/input/d-fire')            # must contain <split>/images and <split>/labels
HPWREN_DIR = Path('/kaggle/input/wildfire-smoke')   # optional, set to None if not attached
OUT_DIR = Path('/kaggle/working/m2_smoke')
for p in (DFIRE_DIR, HPWREN_DIR):
    if p is not None:
        print(p, 'OK' if p.exists() else 'MISSING')

In [ ]:
# 3) Build the smoke-only dataset + leakage report (fails loudly if a fire event crosses splits)
args = f'--dfire {DFIRE_DIR} --out {OUT_DIR}' + (f' --hpwren {HPWREN_DIR}' if HPWREN_DIR else '')
!python -m src.detection.prepare_dataset {args}

In [ ]:
# 4) Sanity check: smoke class id found from per-class counts must match the D-Fire README
import json
report = json.loads((OUT_DIR / 'split_report.json').read_text())
print(json.dumps(report['sources'], indent=2))
print(json.dumps(report['splits'], indent=2))
assert report['leakage']['leakage_free']

In [ ]:
# 5) Train (YOLOv8s pretrained on COCO, early stopping on val mAP)
!python -m src.detection.train --data {OUT_DIR}/data.yaml --model yolov8s.pt --epochs 50 --imgsz 640 --batch 16 --device 0

In [ ]:
# 6) Evaluate on the TEST split + latency with and without SAHI tiling -> artifacts/m2_metrics.json
!python -m src.detection.evaluate --weights runs/m2/yolo_smoke/weights/best.pt --data {OUT_DIR}/data.yaml --device 0
print(open('artifacts/m2_metrics.json').read())

In [ ]:
# 7) Visual check: tiled predictions on a few test frames
import numpy as np, matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
from src.detection.detector import SmokeDetector
det = SmokeDetector('runs/m2/yolo_smoke/weights/best.pt', device='cuda:0')
imgs = sorted((OUT_DIR / 'images' / 'test').glob('*'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
for ax, p in zip(axes.flat, imgs):
    frame = np.asarray(Image.open(p).convert('RGB'))
    ax.imshow(frame); ax.set_title(p.name[:40]); ax.axis('off')
    for d in det.predict(frame):
        x1, y1, x2, y2 = d.box
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='orange', lw=2))
        ax.text(x1, y1 - 4, f'{d.confidence:.2f}', color='orange')
plt.tight_layout(); plt.savefig('artifacts/m2_test_predictions.png', dpi=80)

In [ ]:
# 8) Export files to download from the Output tab
import shutil
shutil.copy('runs/m2/yolo_smoke/weights/best.pt', '/kaggle/working/m2_smoke_best.pt')
shutil.copy('artifacts/m2_metrics.json', '/kaggle/working/m2_metrics.json')
shutil.copy(OUT_DIR / 'split_report.json', '/kaggle/working/m2_split_report.json')
print('Download m2_smoke_best.pt and put it in artifacts/ to serve it with the API.')